In [ ]:
!pip install openmeteo_requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 707.8/707.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.7 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [ ]:
from datetime import datetime
import openmeteo_requests

class IncreaseSpeed:
    def __init__(self, current_speed: int, max_speed: int, step: int = 10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed >= self.max_speed:
            raise StopIteration
        self.current_speed = min(self.current_speed + self.step, self.max_speed)
        return self.current_speed


class DecreaseSpeed:
    def __init__(self, current_speed: int, min_speed: int = 0, step: int = 10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed <= self.min_speed:
            raise StopIteration
        self.current_speed = max(self.current_speed - self.step, self.min_speed)
        return self.current_speed


class Car():
  _tot_cars = 0
  '''
  Car class.
  Has a class variable for counting total amount of cars on the road (increased by 1 upon instance initialization).

  Constructor params:
    max_speed: a maximum possible speed, km/h
    current_speed: current speed, km/h (0 by default)
    state: reflects if the Car is in the parking or on the road

  Methods:
    accelerate: increases the speed using IncreaseSpeed() iterator either once or gradually to the upper_border
    brake: decreases the speed using DecreaseSpeed() iterator either once or gradually to the lower_border
    parking: if the Car is not already in the parking, removes the Car from the road
    total_cars: show the total amount of cars on the road
    show_weather: shows the current weather conditions
  '''

  def __init__(self, max_speed: int, current_speed=0):
    self.max_speed = max_speed
    self.current_speed = current_speed
    self._on_road = current_speed > 0
    if self._on_road:
      Car._tot_cars += 1


  def accelerate(self, upper_border=None, step=10):
    # check for state
    # create an instance of IncreaseSpeed iterator
    # check if smth passed to upper_border and if it is valid speed value
    # if True, increase the speed gradually iterating over your increaser until upper_border is met
    # print a message at each speed increase
    # else increase the speed once
    # return the message with current speed
    speed_before = self.current_speed

    if upper_border is not None and 0 < upper_border <= self.max_speed:
        increaser = IncreaseSpeed(self.current_speed, upper_border, step)
        for new_speed in increaser:
            self.current_speed = new_speed
            print(f"INFO: Speed increased by {step}")
    else:
        increaser = IncreaseSpeed(self.current_speed, self.max_speed, step)
        self.current_speed = next(increaser, self.current_speed)


    if not self._on_road and self.current_speed > 0:
        self._on_road = True
        Car._tot_cars += 1
    print(f"The speed of this car has been increased from {speed_before} to {self.current_speed}")

  def brake(self, lower_border=None, step=10):
    # create an instance of DecreaseSpeed iterator
    # check if smth passed to lower_border and if it is valid speed value
    # if True, decrease the speed gradually iterating over your decreaser until lower_border is met
    # print a message at each speed decrease
    # else increase the speed once
    # return the message with current speed
    speed_before = self.current_speed

    if lower_border is not None and lower_border >= 0:
        decreaser = DecreaseSpeed(self.current_speed, lower_border, step)
        for new_speed in decreaser:
            self.current_speed = new_speed
            print(f"INFO: Speed decreases by {step}")
    else:
        decreaser = DecreaseSpeed(self.current_speed, 0, step)
        self.current_speed = next(decreaser, self.current_speed)
    print(f"INFO: The speed of this car has been decreased from {speed_before} to {self.current_speed}")
  # the next three functions you have to define yourself
  # one of the is class method, one - static and one - regular method (not necessarily in this order, it's for you to think)

  def parking(self):
    # gets car off the road (use state and class variable)
    # check: should not be able to move the car off the road if it's not there
    if not self._on_road:
      print("Already parked")
      return
    self.brake(0)
    self._on_road = False
    Car._tot_cars -=1

  @classmethod
  def total_cars(cls):
    # displays total amount of cars on the road
    return cls._tot_cars
  @staticmethod
  def show_weather():
    # displays weather conditions
    client = openmeteo_requests.Client()
    params = {
        "latitude": 59.9386,
        "longitude": 30.3141,
        "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
        "wind_speed_unit": "ms",
        "timezone": "Europe/Moscow"
    }
    response = client.weather_api("https://api.open-meteo.com/v1/forecast", params=params)[0]
    cur = response.Current()
    print(f"Current temperature: {round(cur.Variables(0).Value(), 1)} C")
    print(f"Current apparent_temperature: {round(cur.Variables(1).Value(), 1)} C")
    print(f"Current rain: {cur.Variables(2).Value()} mm")
    print(f"Current wind_speed: {round(cur.Variables(3).Value(), 1)} m/s")

In [ ]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

# >> Total cars on road: 2



Total cars on road: 2


In [ ]:
car1.accelerate(100)

# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: The speed of this car has been increased from 20 to 100



INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
The speed of this car has been increased from 20 to 100


In [ ]:
car2.accelerate(50)

# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: The speed of this car has been increased from 30 to 50



INFO: Speed increased by 10
INFO: Speed increased by 10
The speed of this car has been increased from 30 to 50


In [ ]:
print("Speed of car 1:", car1.current_speed)

# >> Speed of car 1: 100

print("Speed of car 2:", car2.current_speed)

# >> Speed of car 2: 50



Speed of car 1: 100
Speed of car 2: 50


In [ ]:
car1.brake(10)

# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: The speed of this car has been decreased from 100 to 10



INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [ ]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: The speed of this car has been decreased from 50 to 0
# >> Total cars on road: 2
# >> INFO: The speed of this car has been decreased from 0 to 0 # parking method ensures the speed is 0 in the end
# >> Parking the car...
# >> Total cars on road: 1

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Total cars on road: 1


In [ ]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

# >> INFO: The speed of this car has been increased from 0 to 80
# >> Current temperature: -0.0 C
# >> Current apparent_temperature: 14.0 C
# >> Current rain: 250.0 mm
# >> Current wind_speed: 0.0 m/s
# >> Total cars on road: 2


INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
INFO: Speed increased by 10
The speed of this car has been increased from 0 to 80
Current temperature: 3.3 C
Current apparent_temperature: -0.9 C
Current rain: 0.0 mm
Current wind_speed: 4.1 m/s
Total cars on road: 2


In [ ]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

# >> INFO: Speed increases by 10
# >> INFO: The speed of this car has been increased from 0 to 10
# >> Total cars on road: 3


INFO: Speed increased by 10
The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [ ]:
Car.show_weather()

# >> Current temperature: -0.0 C
# >> Current apparent_temperature: 14.0 C
# >> Current rain: 250.0 mm
# >> Current wind_speed: 0.0 m/s

Current temperature: 3.3 C
Current apparent_temperature: -0.9 C
Current rain: 0.0 mm
Current wind_speed: 4.1 m/s
